# 23. 발표 이후 피드백: 단일 곡 AI 생성 여부 판별

사용자가 제공한 서로 다른 오디오 두 곡을 기존 프로젝트의 최종 튜닝 모델에 입력한다. 사용자 확인 기준으로 한 곡은 AI 생성곡이고, 다른 한 곡은 별개의 AI 생성 샘플에 대응하는 인간 원곡이다. 두 입력은 서로 직접 대응하는 AI/원곡 쌍이 아니다. 파일명이나 메타데이터는 특징으로 사용하지 않고, PCM 오디오만 사용한다. 모델은 10초 구간별 점수를 낸 뒤 곡 단위 평균으로 최종 판정한다.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_FILES = [
    PROJECT_ROOT / "data/raw/inference_inputs/01.wav",
    PROJECT_ROOT / "data/raw/inference_inputs/02.mp3",
]
OUTPUT_DIR = PROJECT_ROOT / "results/post_presentation_feedback"

from src.single_track_inference import run_inference, save_results
results = run_inference(INPUT_FILES, PROJECT_ROOT)
save_results(results, OUTPUT_DIR)
print("저장 위치:", OUTPUT_DIR.relative_to(PROJECT_ROOT))

KeyboardInterrupt: 

## 입력 QC와 추론 규칙

- 24 kHz mono로 디코드하고, 길이가 30초 이상이면 시작·중간·끝, 20초 이상 30초 미만이면 시작·끝의 10초를 사용한다.
- `Logistic Regression`, `RBF-SVM`, `Log-Mel CNN`, `Frozen MERT + LR` 네 모델을 비교한다.
- Validation에서 고정한 track threshold를 적용한다. SVM은 확률이 아닌 decision margin이므로 숫자를 확률(%)로 해석하지 않는다.

In [ ]:
display(results["input_metadata"][["file_name", "duration_sec", "source_sample_rate_hz", "channels", "bytes", "average_bitrate_bps", "sha256"]])
display(results["track_predictions"][["file_name", "model", "n_segments", "ai_score", "track_threshold", "prediction"]].round(6))
display(results["consensus"])

,file_name,duration_sec,source_sample_rate_hz,channels,bytes,average_bitrate_bps,sha256
0,01.wav,29.940000,32000,1,1916204,512011.756847,518b1ad6174bb9462a7ce8bf795e7a7f0a919762f29933...
2,02.mp3,29.976583,44100,2,720629,192317.848098,f9a3cfa7a4fddad7c5f74b3856cb650a0879156a64a326...


,file_name,model,n_segments,ai_score,track_threshold,prediction
0,01.wav,Logistic Regression,2,0.901001,0.563456,AI 생성
1,02.mp3,Logistic Regression,2,0.153920,0.563456,인간 제작
2,01.wav,RBF-SVM,2,1.540211,0.661505,AI 생성
3,02.mp3,RBF-SVM,2,-1.068091,0.661505,인간 제작
4,01.wav,Log-Mel CNN,2,0.853339,0.241069,AI 생성
5,02.mp3,Log-Mel CNN,2,0.005362,0.241069,인간 제작
6,01.wav,Frozen MERT + LR,2,0.989051,0.588751,AI 생성
7,02.mp3,Frozen MERT + LR,2,0.113829,0.588751,인간 제작


,file_name,ai_votes,model_count,consensus_prediction
0,01.wav,4,4,AI 생성
1,02.mp3,0,4,인간 제작


In [ ]:
segment_view = results["segment_predictions"][["file_name", "segment_role", "start_sec", "model", "score", "segment_threshold", "segment_prediction"]].copy()
display(segment_view.round(6))

,file_name,segment_role,start_sec,model,score,segment_threshold,segment_prediction
0,01.wav,start,0.000000,Logistic Regression,0.883259,0.567690,AI 생성
1,01.wav,end,19.940000,Logistic Regression,0.918744,0.567690,AI 생성
2,02.mp3,start,0.000000,Logistic Regression,0.020718,0.567690,인간 제작
3,02.mp3,end,19.976583,Logistic Regression,0.287123,0.567690,인간 제작
4,01.wav,start,0.000000,RBF-SVM,1.681340,0.584756,AI 생성
5,01.wav,end,19.940000,RBF-SVM,1.399082,0.584756,AI 생성
6,02.mp3,start,0.000000,RBF-SVM,-1.957585,0.584756,인간 제작
7,02.mp3,end,19.976583,RBF-SVM,-0.178597,0.584756,인간 제작
8,01.wav,start,0.000000,Log-Mel CNN,0.737370,0.142173,AI 생성
9,01.wav,end,19.940000,Log-Mel CNN,0.969307,0.142173,AI 생성


## 정답 구성과 판정 결과

사용자 확인 정답은 `01.wav`가 **AI 생성곡**, `02.mp3`가 **별개의 AI 생성 샘플에 대응하는 인간 원곡**이다. 두 입력은 실제 디코딩 길이가 30초보다 조금 짧아 각각 시작·끝 2개 구간으로 판정했다. 네 모델 모두 `01.wav`를 **AI 생성**, `02.mp3`를 **인간 제작**으로 판정해 두 곡의 정답과 모두 일치했다.

이번 결과는 곡 기준 2/2, 모델별 곡 판정 기준 8/8이 정답과 일치한 작동 확인이다. 두 표본만으로 일반화 성능이나 정확도를 추정할 수 없으며, `02.mp3`의 원본 FMA 곡은 학습 split에 포함되어 있어 unseen 성능의 근거로 사용할 수 없다. `01.wav`도 프로젝트가 이미 다룬 MusicGen 생성 음원이며, 원시 매니페스트의 파일명 충돌 때문에 정확한 곡명과 split은 확정할 수 없다. 따라서 이 입력 역시 미관측 생성기에 대한 평가가 아니다. 자세한 모델별 점수와 입력 메타데이터는 위 실행 결과와 `results/post_presentation_feedback/`에 저장했다.